# Google Trends -- Trinidad and Tobago

Last update: 2026-10-19. Ported from the Bolivia pipeline
(`2026_NWCST_BOL_Workshop/code/02_gtrends.ipynb`); the previous version (manual
CSV merge only) is in the private project repository.

Writes `raw/gtrends.csv`. Terms come from `data/TT_GTrends_Keywords.csv`
(`keyword`, `geo`, `column`; an empty `geo` means worldwide). **Run with the
`nwcst` kernel** -- it has `trendspy`.

**1. One term per query, never batches.** Google scales results 0-100 *within a
query*, so in a batch every term is scaled to the most-searched one and the small
ones collapse to near-constant integers. Queried alone, each spans the full range.

**2. Rate limits.** `trendspy` is unofficial. Bolivia measured: at 1 s between
queries the ~42nd call returns 429; at 2.5 s, ~33 terms over 2004-today. After a
429 Google keeps blocking for tens of minutes. So the notebook pauses ~10 s per
term, **saves after every term**, and on a re-run **skips the terms already
current** -- interrupted runs resume where they stopped, even the next day.

**3. Updating means re-downloading.** Each download is rescaled 0-100 over its own
window, so a new month cannot be appended to an old series: the whole term is
replaced. `data/gtrends_log.csv` records the last month each term was fetched
through; a term is current when that month is the last complete one.

**Part B** (bottom) is the fallback when Google blocks: it prints a link per
missing term; downloaded CSVs go in `raw/gtrends_manual/` and the last cell merges
them. That subfolder is not read by `2_data_build` (it globs `raw/*.csv` only).

In [ ]:
# ===========================================================================
# CONFIG LOADER  --  edit config.py, NOT this cell.
# ===========================================================================
import os, sys, importlib.util

CODE_DIR = os.getcwd()
_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)

PATH_RAW  = P["raw"]      # gtrends.csv is written here
PATH_DATA = P["data"]     # keyword list and fetch log live here

print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE})")
print(f"         raw  : {PATH_RAW}")
print(f"         data : {PATH_DATA}")

In [ ]:
# trendspy is imported in the download cell, so Part B works without it.
import re, time, random, unicodedata, warnings
from datetime import date

import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ---- parameters ------------------------------------------------------------
GT_START       = cfg.GT_START                       # 2004-01-01 = "all" on the website
GT_END         = date.today().strftime("%Y-%m-%d")
GT_TIMEFRAME   = f"{GT_START} {GT_END}"
GT_PAUSE       = 10.0     # seconds between queries (Bolivia: 2.5 s 429s at term ~34)
GT_PAUSE_MIN   = GT_PAUSE - 0.1                     # each term draws its pause from
GT_PAUSE_MAX   = GT_PAUSE + 0.5                     # this range, centred on GT_PAUSE
GT_MIN_NONZERO = 0.15     # a NEW term with fewer non-zero months is not saved
GT_REFRESH     = False    # True = re-download every term even if current
# Trinidad's gtrends.csv has always stored a zero month as empty, and meta.csv's
# start dates assume it (2_data_build's gtrends_mean averages the non-empty terms).
# Bolivia keeps zeros as zeros. False here would change the build's input.
GT_ZERO_AS_NA  = True

GT_CSV     = os.path.join(PATH_RAW, "gtrends.csv")
GT_LOG     = os.path.join(PATH_DATA, "gtrends_log.csv")
GT_MANUAL  = os.path.join(PATH_RAW, "gtrends_manual")

# the last COMPLETE month: Google's current month is partial (isPartial)
THIS_MONTH = pd.Timestamp(date.today()).to_period("M").to_timestamp()
LAST_FULL  = (pd.Timestamp(date.today()).to_period("M") - 1).to_timestamp()


def slug(s):
    """'First Citizens Bank' -> 'first_citizens_bank'."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_")


def save():
    """Write `series` (column -> Series) to gtrends.csv, keeping `order` first."""
    if not series:
        return None
    cols = [c for c in order if c in series] + [c for c in series if c not in order]
    df = pd.DataFrame({c: series[c] for c in cols})
    df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
    df.index.name = "date"
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df = df[df.index < THIS_MONTH]                 # the current month is partial
    if GT_ZERO_AS_NA:
        df = df.mask(df == 0)
    df.to_csv(GT_CSV)
    return df


def log_fetch(col, s, source):
    """Record in data/gtrends_log.csv the last complete month `col` was fetched through."""
    log.loc[col, ["through", "fetched_on", "source"]] = [
        pd.Timestamp(s[s.index < THIS_MONTH].index.max()).strftime("%Y-%m-%d"),
        date.today().isoformat(), source]
    log.rename_axis("column").to_csv(GT_LOG)


print(f"period={GT_TIMEFRAME}  last complete month={LAST_FULL.date()}")

## Step 1: terms and what is already current

In [ ]:
kw = pd.read_csv(os.path.join(PATH_DATA, cfg.GT_KEYWORDS), dtype=str).fillna("")
kw["keyword"] = kw["keyword"].str.strip()
kw = kw[kw["keyword"] != ""].drop_duplicates(subset=["keyword", "geo"])
# a row with no column name gets the old naming rule: gtrends_<term>_<geo or worldwide>
_auto = "gtrends_" + kw["keyword"].map(slug) + "_" + kw["geo"].str.lower().replace("", "worldwide")
kw["column"] = kw["column"].where(kw["column"] != "", _auto)

have = (pd.read_csv(GT_CSV, index_col="date", parse_dates=["date"])
        if os.path.exists(GT_CSV) else pd.DataFrame())
log = (pd.read_csv(GT_LOG, parse_dates=["through"]).set_index("column")
       if os.path.exists(GT_LOG) else pd.DataFrame(columns=["through", "fetched_on", "source"]))

def is_current(col):
    return (not GT_REFRESH and col in have.columns and col in log.index
            and pd.Timestamp(log.at[col, "through"]) >= LAST_FULL)

kw["current"] = kw["column"].map(is_current)
todo = kw[~kw["current"]]

print(f"{len(kw)} terms | {int(kw['current'].sum())} current | {len(todo)} to download")
if len(have):
    print(f"gtrends.csv: {have.shape[1]} series, {have.index.min().date()} to {have.index.max().date()}")
kw

## Step 2: download with `trendspy`

In [ ]:
from trendspy import Trends              # if missing:  pip install trendspy

tr = Trends(request_delay=GT_PAUSE)
series = {c: have[c] for c in have.columns}     # replaced term by term
order  = list(have.columns)
thin, failed = [], []

for i, r in enumerate(todo.itertuples(), 1):
    term, geo, col = r.keyword, r.geo, r.column
    tr.request_delay = random.triangular(GT_PAUSE_MIN, GT_PAUSE_MAX, GT_PAUSE)
    where = geo or "worldwide"
    try:
        d = tr.interest_over_time([term], geo=geo, timeframe=GT_TIMEFRAME)
        c = [c for c in d.columns if c != "isPartial"][0]
        s = pd.to_numeric(d[c], errors="coerce")
        s.index = pd.to_datetime(d.index).tz_localize(None).normalize()
        share = float((s > 0).mean())

        if col not in have.columns and share < GT_MIN_NONZERO:
            thin.append(term)
            print(f"  [{i:2d}/{len(todo)}] {term:22} {where:9} thin ({share:.0%} non-zero) -- not saved")
            continue
        series[col] = s
        save()                                      # <- saved after every term
        log_fetch(col, s, "trendspy")
        print(f"  [{i:2d}/{len(todo)}] {term:22} {where:9} ok  ({share:.0%} non-zero)")

    except Exception as e:
        failed.append(term)
        print(f"  [{i:2d}/{len(todo)}] {term:22} {where:9} !! {type(e).__name__}: {str(e)[:80]}")
        if "429" in str(e) or "Too Many" in str(e):
            print("      Google rate limit. Everything so far is saved. Wait (tens of")
            print("      minutes) and re-run this cell -- it resumes here -- or use Part B.")
            break

gt = save()
print()
if gt is not None:
    print(f"gtrends.csv: {gt.shape[0]} rows x {gt.shape[1]} series | "
          f"{gt.index.min().date()} to {gt.index.max().date()}")
if thin:
    print(f"too thin, not saved: {thin}")
if failed:
    print(f"not downloaded this run: {failed}")

## Step 3: a look

In [ ]:
gt = pd.read_csv(GT_CSV, index_col="date", parse_dates=["date"]).sort_index()

show = [c for c in ["gtrends_amazon_tt", "gtrends_kfc_tt", "gtrends_caribbean_jobs_tt",
                    "gtrends_tobago_tt", "gtrends_rbc_tt"] if c in gt.columns]

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
gt[show].plot(ax=ax[0], lw=1)
ax[0].set_title(f"Google Trends, {cfg.COUNTRY} -- monthly search interest")
ax[0].set_ylabel("0-100"); ax[0].legend(fontsize=8, ncol=3)
gt[show].rolling(12, min_periods=6).mean().plot(ax=ax[1], lw=1.2)
ax[1].set_ylabel("12-month mean"); ax[1].legend().remove()
plt.tight_layout(); plt.show()

print(f"{gt.shape[1]} series, {gt.index.min().date()} to {gt.index.max().date()}")
print()
print("last month with data, per series (stale first):")
print(gt.apply(lambda s: s.last_valid_index()).sort_values().head(8).dt.date.to_string())

---
# Part B: manual download

When Google keeps returning 429, download by hand. The next cell lists the terms
that are still not current and prints the exact Google Trends link for each
(same period and country as the scraper). Open it, download the time-series CSV
(download button, top right of the chart), save it **unrenamed** in
`raw/gtrends_manual/`, then run the last cell.

In [ ]:
from urllib.parse import quote

os.makedirs(GT_MANUAL, exist_ok=True)
_cur = (pd.read_csv(GT_LOG, parse_dates=["through"]).set_index("column")
        if os.path.exists(GT_LOG) else pd.DataFrame(columns=["through"]))
_thin = globals().get("thin", [])
pending = kw[[not (c in _cur.index and pd.Timestamp(_cur.at[c, "through"]) >= LAST_FULL)
              and k not in _thin for c, k in zip(kw["column"], kw["keyword"])]]


def trends_link(term, geo):
    period = "all" if GT_START == "2004-01-01" else f"{GT_START}%20{GT_END}"
    return (f"https://trends.google.com/trends/explore?date={period}"
            + (f"&geo={geo}" if geo else "") + f"&q={quote(term)}")


if not len(pending):
    print("Nothing pending: every term is current.")
else:
    print(f"{len(pending)} of {len(kw)} terms not current. Save the CSVs in:\n  {GT_MANUAL}\n")
    for r in pending.itertuples():
        print(f"  {r.keyword}  ({r.geo or 'worldwide'})")
        print(f"    {trends_link(r.keyword, r.geo)}")

## Merge the manual downloads into `gtrends.csv`

Reads both export formats Google has used:

| file | format |
|---|---|
| `multiTimeline*.csv` | old page: a title line, then `Month`, `term: (Country)` |
| `time_series_<GEO>_*.csv` | current page: `Time` and the term |

Each column is matched to its row in the keyword list (by term, and by country
when the same term is listed twice) and takes that row's `column` name, so a
manual series lands exactly where the scraped one would. A manual series
**replaces** that term in `gtrends.csv`; everything else is kept. The downloaded
files are left in `raw/gtrends_manual/`.

In [ ]:
import glob

multi = sorted(glob.glob(os.path.join(GT_MANUAL, "*ultiTimeline*.csv")))
single = sorted(glob.glob(os.path.join(GT_MANUAL, "time_series_*.csv")))
print(f"{len(multi)} multiTimeline file(s), {len(single)} time_series file(s)")


def target_column(term, geo_hint):
    """Column name for a downloaded term, from the keyword list."""
    rows = kw[kw["keyword"].map(slug) == slug(term)]
    if len(rows) > 1:
        rows = rows[rows["geo"].str.upper() == geo_hint.upper()] if geo_hint else rows[rows["geo"] == ""]
    if len(rows) == 1:
        return rows["column"].iloc[0]
    return "gtrends_" + slug(term) + "_" + (geo_hint.lower() or "worldwide")


pieces = []
for f in multi:
    d = pd.read_csv(f, skiprows=1).rename(columns={"Month": "date"})
    for c in d.columns[1:]:
        m = re.match(r"(.*?):\s*\((.*)\)\s*$", str(c))
        term, place = (m.group(1), m.group(2)) if m else (str(c), "")
        geo = "" if place.lower() == "worldwide" else cfg.CODE
        pieces.append(d.set_index("date")[c].rename(target_column(term, geo)))
for f in single:
    m = re.match(r"time_series_([A-Za-z]+)_", os.path.basename(f))
    geo = m.group(1).upper() if m else ""
    geo = "" if geo in ("GLB", "WORLDWIDE", "WW") else geo
    d = pd.read_csv(f).rename(columns={"Time": "date"})
    for c in d.columns[1:]:
        pieces.append(d.set_index("date")[c].rename(target_column(c, geo)))

if not pieces:
    print("No manual files in raw/gtrends_manual/.")
else:
    manual = pd.concat(pieces, axis=1)
    manual.index = pd.to_datetime(manual.index, errors="coerce")
    manual = manual[manual.index.notna()]
    manual = manual.loc[:, ~manual.columns.duplicated(keep="last")]
    # Google writes "<1" for minimal interest: that is ~0, not missing
    manual = manual.replace("<1", 0).apply(pd.to_numeric, errors="coerce")

    series = ({c: s for c, s in pd.read_csv(GT_CSV, index_col="date", parse_dates=["date"]).items()}
              if os.path.exists(GT_CSV) else {})
    order = list(series)
    for c in manual.columns:
        series[c] = manual[c]
        log_fetch(c, manual[c].dropna(), "manual")
    out = save()
    print(f"-> {GT_CSV}: {out.shape[0]} rows x {out.shape[1]} series | "
          f"{out.index.min().date()} to {out.index.max().date()}")
    print("merged:", list(manual.columns))